# **Leyendo la base de datos**

In [ ]:
import pandas as pd
dataset_entrenamiento = pd.read_csv('../data/train_clean.csv')
dataset_prueba = pd.read_csv('../data/test_clean.csv')
dataset_oot = pd.read_csv('../data/oot_clean.csv')

# **Creando X & Y**

In [ ]:
X_entrenamiento = dataset_entrenamiento.drop(columns=['es_fraude'])
y_entrenamiento = dataset_entrenamiento['es_fraude']

X_prueba = dataset_prueba.drop(columns=['es_fraude'])
y_prueba = dataset_prueba['es_fraude']

X_oot = dataset_oot.drop(columns=['es_fraude'])
y_oot = dataset_oot['es_fraude']

print("Dimensiones:")
print(f"Entrenamiento -> X: {X_entrenamiento.shape}, y: {y_entrenamiento.shape}")
print(f"Prueba        -> X: {X_prueba.shape}, y: {y_prueba.shape}")
print(f"OOT           -> X: {X_oot.shape}, y: {y_oot.shape}")

In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# **Hiperparámetros**

In [ ]:
peso_positivos = (y_entrenamiento == 0).sum() / (y_entrenamiento == 1).sum()

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-2, 5.0, log=True),
        'scale_pos_weight': peso_positivos,
        'n_estimators': 500,
        'random_state': 42,
        'eval_metric': 'auc',
        'early_stopping_rounds': 30,
        'tree_method': 'hist',
        'n_jobs': -1,
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_entrenamiento, y_entrenamiento):
        X_tr, X_val = X_entrenamiento.iloc[train_idx], X_entrenamiento.iloc[val_idx]
        y_tr, y_val = y_entrenamiento.iloc[train_idx], y_entrenamiento.iloc[val_idx]

        modelo = XGBClassifier(**params)
        modelo.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = modelo.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, preds))

    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("Mejores hiperparámetros:", study.best_params)
print("Mejor AUC:", study.best_value)

**Los mejores hiperparámetros fueron:**

**max_depth: 7**
Profundidad máxima de cada árbol. 7 es un valor intermedio-alto: el modelo necesita capturar interacciones no triviales entre variables para separar fraude de no-fraude, pero no se fue a los extremos (9-10) que suelen sobreajustar en datasets desbalanceados.

**min_child_weight: 10**
El valor más alto permitido en la rejilla (rango era 1-10). Esto es una señal importante: el modelo quiere ser exigente al decidir splits, exigiendo que cada nodo hijo tenga bastante peso de muestras antes de dividir. Es coherente con max_depth=7, compensa la profundidad con más regularización, evitando que los árboles memoricen ruido de los pocos casos de fraude.

**gamma: 3.80**
También cerca del techo (rango 0-5). Un split solo se acepta si reduce la pérdida en al menos esa cantidad. Gamma alto + min_child_weight alto = el modelo penaliza fuerte los splits "débiles". Ambos hiperparámetros apuntando hacia el lado conservador confirma que sin esa regularización, el modelo probablemente sobreajustaría de forma severa.

**learning_rate: 0.078**
Ni muy agresivo ni muy lento, dentro del rango 0.01-0.3. Con ~500 árboles de techo y early stopping, este valor permite que el modelo converja de forma gradual y estable sin necesitar miles de iteraciones.

**subsample: 0.788**
Cada árbol se entrena con ~79% de las filas, muestreadas al azar. Ayuda a que los árboles no sean todos idénticos y reduce varianza.

**colsample_bytree: 0.91**
Cada árbol usa ~91% de las columnas. Al ser un valor alto, sugiere que la mayoría de tus variables aportan señal útil — el modelo no necesita descartar muchas columnas para mejorar el rendimiento.

**reg_alpha: 1.14 / reg_lambda: 0.60**
Regularización L1 y L2 respectivamente. Ambos en rango moderado (ni cerca de 0, ni en el techo de 5). Interesante que reg_alpha > reg_lambda — el modelo se beneficia un poco más de L1 (que puede llevar coeficientes/splits menos relevantes a cero) que de L2 puro.

---

**Lectura general**
Los valores de min_child_weight y gamma empujando hacia sus topes es la señal más clara del set: el modelo necesitó bastante regularización estructural para no sobreajustar. Esto es esperable en un problema de fraude con clases desbalanceadas.

**Mejor AUC:**

0.9943596567126362

# **Modelo**

In [ ]:
mejores_params = {
    'max_depth': 7,
    'min_child_weight': 10,
    'gamma': 3.8043933824847453,
    'learning_rate': 0.07829783576408142,
    'subsample': 0.7880728574863048,
    'colsample_bytree': 0.9099292774168041,
    'reg_alpha': 1.143515072805884,
    'reg_lambda': 0.5983681497070954,
    'scale_pos_weight': scale_pos_weight,
    'n_estimators': 500,
    'random_state': 42,
    'eval_metric': 'auc',
    'early_stopping_rounds': 50,
    'tree_method': 'hist',
    'n_jobs': -1,
}

xgboost = XGBClassifier(**mejores_params)

xgboost.fit(
    X_entrenamiento, y_entrenamiento,
    eval_set=[(X_prueba, y_prueba)],
    verbose=False
)

# **Umbral óptimo**

In [ ]:
import numpy as np
from sklearn.metrics import fbeta_score, classification_report, confusion_matrix, roc_auc_score, f1_score

y_proba_prueba_xgb = xgboost.predict_proba(X_prueba)[:, 1]

mejor_umbral_xgb = 0.5
mejor_f2_xgb = 0.0

for umbral in np.arange(0.01, 1.0, 0.01):
    prediccion_temporal = (y_proba_prueba_xgb >= umbral).astype(int)
    f2_temporal = fbeta_score(y_prueba, prediccion_temporal, beta=2, zero_division=0)

    if f2_temporal > mejor_f2_xgb:
        mejor_f2_xgb = f2_temporal
        mejor_umbral_xgb = umbral

print(f"Umbral ganador: {mejor_umbral_xgb:.2f}")
print(f"Mejor F2 en prueba: {mejor_f2_xgb:.4f}")

# **Desempeño**

In [ ]:
def evaluar_con_umbral(y_real, y_proba, umbral, nombre_set):
    pred = (y_proba >= umbral).astype(int)
    print(f"\n{'='*50}")
    print(f"Resultados en {nombre_set} (umbral = {umbral:.2f})")
    print(f"{'='*50}")
    print(f"AUC-ROC: {roc_auc_score(y_real, y_proba):.4f}")
    print(f"F2-score: {fbeta_score(y_real, pred, beta=2):.4f}")
    print(f"\n{classification_report(y_real, pred, target_names=['No fraude', 'Fraude'], digits=5)}")
    print("Matriz de confusión:")
    print(confusion_matrix(y_real, pred))

evaluar_con_umbral(y_prueba, y_proba_prueba_xgb, mejor_umbral_xgb, "Prueba")
y_proba_oot_xgb = xgboost.predict_proba(X_oot)[:, 1]
evaluar_con_umbral(y_oot, y_proba_oot_xgb, mejor_umbral_xgb, "OOT")